# Evaluation — cost per stage and quality gained

Reads the sidecar tables and the codecarbon log. Quality follows the plan's four axes (completeness, consistency, conformance, linkage), each before/after, and §8 ties them to energy as marginal gain per Wh.

In [1]:
from pathlib import Path

import polars as pl
import plotly.graph_objects as go
from plotly.subplots import make_subplots

from mds_norm.parsers.parse_dates import parse_date
from mds_data_model.introspection import date_fields

In [2]:
import json

from mds_norm.paths import (
    DATA,
    EMISSIONS_CSV as EMISSIONS,
    EVAL_OUT as OUT_DIR,
    EXP_OUT as EXPERIMENTS_PATH,
    FIELD_STATS,
    LLM_RESPONSES,
    PERSON_ANNOTATIONS,
    PERSON_DECISIONS,
    PROBE_CANDIDATES,
    RECORD_PATCHES,
    VOCAB_ANNOTATIONS,
    VOCAB_DECISIONS,
)

OUT_DIR.mkdir(parents=True, exist_ok=True)

ELECTRICITY_GBP_PER_KWH = 0.27  # Ofgem cap, unit rate

# Palette: ordinal ramp, before/after pair, categorical slots, neutral grays
INK, INK2, MUTED = "#0b0b0b", "#52514e", "#898781"
GRID, AXIS, SURFACE = "#e1e0d9", "#c3c2b7", "#fcfcfb"
TIER_RAMP = {0: "#b3d0f6", 1: "#86b6ef", 2: "#5598e7", 3: "#2a78d6", 4: "#1c5cab", 5: "#104281"}
BEFORE, AFTER = "#86b6ef", "#2a78d6"
CAT = ["#2a78d6", "#1baf7a", "#eda100", "#4a3aa7"]
GRAY_LIGHT, GRAY_MID = "#e1e0d9", "#c3c2b7"

SCORE_BINS = 20


def style(fig: go.Figure, title: str, height: int = 380) -> go.Figure:
    fig.update_layout(
        template="none",
        title=dict(text=title, font=dict(size=14, color=INK)),
        font=dict(family='system-ui, "Segoe UI", sans-serif', size=12, color=INK2),
        paper_bgcolor=SURFACE, plot_bgcolor=SURFACE,
        height=height, margin=dict(l=8, r=8, t=48, b=8),
        bargap=0.35, legend=dict(borderwidth=0, traceorder="normal"),
    )
    fig.update_xaxes(gridcolor=GRID, linecolor=AXIS, zerolinecolor=AXIS,
                     ticks="", tickfont=dict(color=MUTED), automargin=True)
    fig.update_yaxes(gridcolor=GRID, linecolor=AXIS, zerolinecolor=AXIS,
                     ticks="", tickfont=dict(color=MUTED), automargin=True)
    return fig


def bin_scores(scores: pl.Series, bins: int = SCORE_BINS) -> pl.DataFrame:
    """Counts per equal-width bin over [0, 1], with bin start and midpoint for plotting"""
    return (
        pl.DataFrame({"score": scores})
        .with_columns(bin=(pl.col("score") * bins).floor().clip(0, bins - 1).cast(pl.Int32))
        .group_by("bin").agg(count=pl.len())
        .with_columns(start=pl.col("bin") / bins, mid=(pl.col("bin") + 0.5) / bins)
        .sort("bin")
    )

## 1. Cost per stage

Every logged run per stage, summed, so this is development compute rather than one pass. Tier 0 and the tier-1 date parse were re-measured once on 2026-07-18 (`evaluation/tier01_rerun.json`).

In [3]:
STAGES = {
    "tier0_standardise": ("tier 0 — standardise", 0),
    "tier1_date_parse": ("dates — EDTF parse", 1),
    "probe_scan_targets": ("probe scan — targets", 1),
    "probe_status_dates": ("probe verify — dates", 1),
    "probe_status_dims_prices": ("probe verify — dims + prices", 1),
    "vocab_index_build": ("vocab — authority indexes", 2),
    "vocab_align_fuzzy": ("vocab — exact + fuzzy", 2),
    # Both spellings map to the same tier-3 slot
    "vocab_align_semantic": ("vocab — semantic", 3),
    # Costed at tier 4, apart from the rung it replaced
    "vocab_rerank_llm": ("vocab — rerank", 4),
    "vocab_atomise_llm": ("vocab — LLM atomiser", 4),
    # Map both termlist project spellings, old and current
    "local_termlists": ("termlists — deterministic rungs", 2),
    "termlist_harvest": ("termlists — deterministic rungs", 2),
    "termlist_delimiter_induction": ("termlists — deterministic rungs", 2),
    "termlist_exact": ("termlists — deterministic rungs", 2),
    "termlist_exact_paren": ("termlists — deterministic rungs", 2),
    "termlist_exact_variant": ("termlists — deterministic rungs", 2),
    "termlist_exact_morph": ("termlists — deterministic rungs", 2),
    "termlist_fuzzy": ("termlists — deterministic rungs", 2),
    "termlist_atomise": ("termlists — deterministic rungs", 2),
    "termlist_semantic": ("termlists — semantic", 3),
    "termlist_llm_atomiser": ("termlists — LLM atomiser", 4),
    "places_pipeline": ("places — parse + TGN cascade", 2),
    "persons_tier2_parse": ("agents — route + parse", 2),
    "record_fixes_mechanical": ("record fixes — mechanical ops", 1),
    "record_fixes_llm": ("record fixes — LLM extraction", 5),
}

emissions = pl.read_csv(EMISSIONS).with_columns(day=pl.col("timestamp").str.slice(0, 10))

# Count each stage's latest day; extraction counts every journalled day
EXTRACTION = "record_fixes_llm"
seeded_days = {
    json.loads(line)["ts"][:10]
    for journal in sorted((DATA / "extraction").glob("progress*.jsonl"))
    for line in journal.read_text(encoding="utf-8").splitlines()
    if line
}
release_days = (
    emissions.group_by("project_name")
    .agg(day=pl.col("day").max())
    .filter(pl.col("project_name") != EXTRACTION)
)
emissions = pl.concat([
    emissions.join(release_days, on=["project_name", "day"], how="semi"),
    emissions.filter((pl.col("project_name") == EXTRACTION) & pl.col("day").is_in(sorted(seeded_days))),
])
unmapped = emissions.filter(
    ~pl.col("project_name").is_in(list(STAGES)))["project_name"].unique().to_list()
if unmapped:
    print(f"unmapped projects excluded from stage totals: {unmapped}")

stage_costs = (
    emissions.filter(pl.col("project_name").is_in(list(STAGES)))
    .with_columns(
        stage=pl.col("project_name").replace_strict({k: v[0] for k, v in STAGES.items()}),
        tier=pl.col("project_name").replace_strict({k: v[1] for k, v in STAGES.items()}),
    )
    .group_by("stage", "tier")
    .agg(
        runs=pl.len(),
        duration_s=pl.col("duration").sum().round(0),
        energy_wh=(pl.col("energy_consumed").sum() * 1e3).round(3),
        co2_g=(pl.col("emissions").sum() * 1e3).round(3),
    )
    .with_columns(cost_gbp=(pl.col("energy_wh") / 1e3 * ELECTRICITY_GBP_PER_KWH).round(5))
    .sort("tier", "energy_wh")
    .select("stage", "tier", "runs", "duration_s", "energy_wh", "co2_g", "cost_gbp")
)

llm_raw = pl.read_parquet(LLM_RESPONSES)
print(f"total: {stage_costs['energy_wh'].sum():.1f} Wh, {stage_costs['co2_g'].sum():.2f} gCO2e, "
      f"£{stage_costs['cost_gbp'].sum():.4f} at £{ELECTRICITY_GBP_PER_KWH}/kWh")
print(f"tier-5 tokens (gpt-oss-20b, local): {llm_raw['prompt_tokens'].sum():,} prompt + "
      f"{llm_raw['completion_tokens'].sum():,} completion over {len(llm_raw):,} records")
display(stage_costs)

unmapped projects excluded from stage totals: ['freeze_metric_weights', 'agent_links', 'persons', 'places_fallback', 'compile_records', 'institutional_priors', 'pattern_induction', 'institutional_fingerprints', 'vocab_align_exact']
total: 8222.3 Wh, 1953.53 gCO2e, £2.2200 at £0.27/kWh
tier-5 tokens (gpt-oss-20b, local): 5,706,389 prompt + 604,468 completion over 5,996 records


stage,tier,runs,duration_s,energy_wh,co2_g,cost_gbp
str,i64,u32,f64,f64,f64,f64
"""tier 0 — standardise""",0,1,46.0,2.377,0.565,0.00064
"""record fixes — mechanical ops""",1,1,4.0,0.089,0.021,0.00002
"""probe verify — dims + prices""",1,1,3.0,0.276,0.066,0.00007
"""dates — EDTF parse""",1,1,104.0,2.662,0.633,0.00072
"""probe verify — dates""",1,1,100.0,16.244,3.859,0.00439
…,…,…,…,…,…,…
"""vocab — semantic""",3,2,205.0,24.008,5.704,0.00648
"""termlists — LLM atomiser""",4,1,10.0,0.809,0.192,0.00022
"""vocab — LLM atomiser""",4,2,2379.0,108.951,25.886,0.02942


In [4]:
d = stage_costs.sort("energy_wh")
fig = go.Figure()
for tier in sorted(d["tier"].unique().to_list()):
    t = d.filter(pl.col("tier") == tier)
    fig.add_bar(
        y=t["stage"].to_list(), x=t["energy_wh"].to_list(), orientation="h",
        name=f"tier {tier}", marker=dict(color=TIER_RAMP[tier]),
        text=[f"{v:,.1f}" for v in t["energy_wh"]], textposition="outside",
        textfont=dict(color=INK2), cliponaxis=False,
        hovertemplate="%{y}<br>%{x:.2f} Wh<extra></extra>",
    )
fig.update_layout(barmode="overlay", legend_traceorder="normal")
fig.update_yaxes(categoryorder="array", categoryarray=d["stage"].to_list())
fig.update_xaxes(title_text="Wh, all logged runs", title_font=dict(color=MUTED))
style(fig, "Energy per stage", height=420)
fig.show()

## 2. Workload and unit cost

Work above tier 0 is deduplicated on distinct values, so occurrences per distinct value is each component's free cost divisor.

In [5]:
vocab_occ = pl.scan_parquet(VOCAB_ANNOTATIONS).select(pl.len()).collect(engine="streaming").item()
vocab_distinct = (pl.scan_parquet(VOCAB_DECISIONS)
                  .select(pl.col("value").n_unique()).collect().item())
person_occ = pl.scan_parquet(PERSON_ANNOTATIONS).select(pl.len()).collect(engine="streaming").item()
person_distinct = (pl.scan_parquet(PERSON_DECISIONS)
                   .select(pl.col("value").n_unique()).collect().item())
probe_spans = pl.scan_parquet(PROBE_CANDIDATES).select(pl.len()).collect().item()
patches = pl.read_parquet(RECORD_PATCHES)
mech_ops = patches.filter(pl.col("sub_component") == "mechanical").height

VOCAB_STAGES = ["vocab — authority indexes", "vocab — exact + fuzzy",
                "vocab — semantic", "vocab — rerank", "vocab — LLM atomiser"]
wh = dict(stage_costs.select("stage", "energy_wh").iter_rows())
probe_wh = wh["probe scan — targets"] + wh["probe verify — dates"] + wh["probe verify — dims + prices"]

workload = pl.DataFrame({
    "component": ["probe scan + verify", "vocabularies (tiers 2–4)", "agents (tier 2)",
                  "record fixes — mechanical", "record fixes — LLM"],
    "distinct_units": [probe_spans, vocab_distinct, person_distinct, mech_ops, len(llm_raw)],
    "unit": ["candidate spans", "distinct values", "distinct values", "patch ops", "records"],
    "occurrences": [probe_spans, vocab_occ, person_occ, mech_ops, len(llm_raw)],
    "energy_wh": [probe_wh, sum(wh[s] for s in VOCAB_STAGES),
                  wh["agents — route + parse"], wh["record fixes — mechanical ops"],
                  wh["record fixes — LLM extraction"]],
}).with_columns(
    occ_per_distinct=(pl.col("occurrences") / pl.col("distinct_units")).round(1),
    wh_per_1k_units=(pl.col("energy_wh") / pl.col("distinct_units") * 1e3).round(4),
    wh_per_1m_occ=(pl.col("energy_wh") / pl.col("occurrences") * 1e6).round(3),
)
workload

component,distinct_units,unit,occurrences,energy_wh,occ_per_distinct,wh_per_1k_units,wh_per_1m_occ
str,i64,str,i64,f64,f64,f64,f64
"""probe scan + verify""",957768,"""candidate spans""",957768,76.468,1.0,0.0798,79.84
"""vocabularies (tiers 2–4)""",608202,"""distinct values""",37199158,1095.125,61.2,1.8006,29.44
"""agents (tier 2)""",542678,"""distinct values""",5451693,2.967,10.0,0.0055,0.544
"""record fixes — mechanical""",691290,"""patch ops""",691290,0.089,1.0,0.0001,0.129
"""record fixes — LLM""",5996,"""records""",5996,7040.791,1.0,1174.248,1.1742e6


## 3. Tier 0 — universal standardisation

`field_stats.parquet` stores the fixed value; `contains_mojibake` counts occurrences repaired.

In [6]:
fs = pl.scan_parquet(FIELD_STATS)
tier0 = fs.select(
    nodes=pl.len(),
    records=pl.col("record_id").n_unique(),
    mojibake_fixed=pl.col("contains_mojibake").sum(),
    contains_html=pl.col("contains_html").sum(),
    contains_url=pl.col("contains_url").sum(),
    contains_email=pl.col("contains_email").sum(),
).collect(engine="streaming")
N_NODES = tier0["nodes"][0]
MOJIBAKE_SHARE = tier0["mojibake_fixed"][0] / N_NODES
tier0

nodes,records,mojibake_fixed,contains_html,contains_url,contains_email
u32,u32,u32,u32,u32,u32
148430656,7438968,19721,629383,4133215,287


## 4. Dates — conformance and consistency (tier 1)

Conformance before is the share of occurrences already in EDTF normal form; after is the share the parser resolves. Consistency is the occurrence share of the 20 most frequent induced patterns.

In [7]:
date_occ = (
    fs.filter(pl.col("field_type").is_in(list(date_fields())))
    .group_by("value", "field_type", "merged_pattern")
    .agg(count=pl.len())
    .collect(engine="streaming")
)

parses, canonical = {}, {}
for v in date_occ["value"].unique().to_list():
    try:
        p = parse_date(v)
    except Exception:
        p = None
    parses[v] = p is not None
    canonical[v] = bool(p) and p.get("value_edtf") == v

date_occ = date_occ.with_columns(
    parses=pl.col("value").map_elements(parses.__getitem__, return_dtype=pl.Boolean),
    already_edtf=pl.col("value").map_elements(canonical.__getitem__, return_dtype=pl.Boolean),
)

date_by_field = (
    date_occ.group_by("field_type")
    .agg(
        occurrences=pl.col("count").sum(),
        distinct=pl.len(),
        before=(pl.col("already_edtf") * pl.col("count")).sum() / pl.col("count").sum(),
        after=(pl.col("parses") * pl.col("count")).sum() / pl.col("count").sum(),
    )
    .sort("occurrences", descending=True)
)

DATE_OCC_TOTAL = date_occ["count"].sum()
DATE_BEFORE = (date_occ["already_edtf"] * date_occ["count"]).sum() / DATE_OCC_TOTAL
DATE_AFTER = (date_occ["parses"] * date_occ["count"]).sum() / DATE_OCC_TOTAL
print(f"all date fields: {DATE_BEFORE:.1%} of occurrences already EDTF → {DATE_AFTER:.1%} parse")
date_by_field

all date fields: 30.1% of occurrences already EDTF → 91.1% parse


field_type,occurrences,distinct,before,after
str,u32,u32,f64,f64
"""spectrum/object_production_dat…",2727641,159623,0.276213,0.844437
"""spectrum/acquisition_date""",1534891,40245,0.349134,0.990468
"""spectrum/field_collection_date""",534405,43018,0.334935,0.960425
"""spectrum/entry_date""",504743,13244,0.055056,0.990623
"""spectrum/associated_date""",352241,30922,0.356131,0.81865
…,…,…,…,…
"""spectrum/right_end_date""",35,1,0.0,1.0
"""spectrum/disposal_date""",27,14,0.037037,1.0
"""spectrum/treatment_begin_date""",16,13,0.3125,1.0


In [8]:
def head_share(frame: pl.DataFrame, col: str, k: int = 20) -> float:
    by_pat = frame.group_by(col).agg(pl.col("count").sum()).sort("count", descending=True)
    return by_pat.head(k)["count"].sum() / by_pat["count"].sum()

pat = date_occ.with_columns(
    pattern_before=pl.col("merged_pattern").fill_null("∅"),
    pattern_after=pl.when(pl.col("parses")).then(pl.lit("EDTF"))
                    .otherwise(pl.col("merged_pattern").fill_null("∅")),
)
CONSISTENCY_BEFORE = head_share(pat, "pattern_before")
CONSISTENCY_AFTER = head_share(pat, "pattern_after")
print(f"head-20 pattern share: {CONSISTENCY_BEFORE:.1%} → {CONSISTENCY_AFTER:.1%}")

d = date_by_field.filter(pl.col("occurrences") > 10_000).sort("occurrences")
fig = go.Figure()
# "after" first, so "before" leads each drawn pair
fig.add_bar(y=d["field_type"].to_list(), x=d["after"].to_list(), orientation="h",
            name="after (parses to EDTF)", marker=dict(color=AFTER),
            hovertemplate="%{y}<br>after %{x:.1%}<extra></extra>")
fig.add_bar(y=d["field_type"].to_list(), x=d["before"].to_list(), orientation="h",
            name="before (already EDTF)", marker=dict(color=BEFORE),
            hovertemplate="%{y}<br>before %{x:.1%}<extra></extra>")
fig.update_layout(barmode="group")
fig.update_xaxes(tickformat=".0%", range=[0, 1],
                 title_text="occurrence share", title_font=dict(color=MUTED))
style(fig, "Date conformance — occurrence share in canonical form", height=440)
fig.update_layout(legend_traceorder="reversed")
fig.show()

head-20 pattern share: 78.3% → 98.0%


## 5. Vocabulary alignment (tiers 2–4)

Occurrence-weighted resolution per field group by rung. Linkage starts at zero, so the resolved share is the improvement.

In [9]:
va = pl.scan_parquet(VOCAB_ANNOTATIONS)

SEGMENTS = ["exact", "exact retry", "fuzzy", "semantic", "llm", "flagged", "deferred"]
SEG_COLOURS = dict(zip(SEGMENTS, [TIER_RAMP[1], TIER_RAMP[2], TIER_RAMP[3], TIER_RAMP[4],
                                  TIER_RAMP[5], GRAY_MID, GRAY_LIGHT]))
segment = (
    pl.when(pl.col("status") != "resolved").then(pl.col("status"))
    .when(pl.col("sub_component") == "exact").then(pl.lit("exact"))
    .when(pl.col("sub_component").str.starts_with("exact_")).then(pl.lit("exact retry"))
    .when(pl.col("sub_component") == "fuzzy").then(pl.lit("fuzzy"))
    .when(pl.col("sub_component") == "semantic").then(pl.lit("semantic"))
    .otherwise(pl.lit("llm"))
)

vocab_seg = (
    va.filter(pl.col("status") != "rejected")
    .with_columns(segment=segment)
    .group_by("group", "segment").agg(occ=pl.len())
    .with_columns(share=pl.col("occ") / pl.col("occ").sum().over("group"))
    .collect(engine="streaming")
)
VOCAB_RESOLVED_SHARE = (
    vocab_seg.filter(~pl.col("segment").is_in(["flagged", "deferred"]))["occ"].sum()
    / vocab_seg["occ"].sum()
)
print(f"occurrences linked across all groups: {VOCAB_RESOLVED_SHARE:.1%}")

order = (
    vocab_seg.filter(~pl.col("segment").is_in(["flagged", "deferred"]))
    .group_by("group").agg(pl.col("share").sum())
    .sort("share")["group"].to_list()
)
order += [g for g in vocab_seg["group"].unique().to_list() if g not in order]

fig = go.Figure()
for seg in SEGMENTS:
    t = vocab_seg.filter(pl.col("segment") == seg)
    fig.add_bar(y=t["group"].to_list(), x=t["share"].to_list(), orientation="h",
                name=seg, marker=dict(color=SEG_COLOURS[seg]),
                customdata=t["occ"].to_list(),
                hovertemplate="%{y} — " + seg + "<br>%{x:.1%} (%{customdata:,} occ)<extra></extra>")
fig.update_layout(barmode="stack", bargap=0.3)
fig.update_yaxes(categoryorder="array", categoryarray=order)
fig.update_xaxes(tickformat=".0%", range=[0, 1],
                 title_text="occurrence share", title_font=dict(color=MUTED))
style(fig, "Vocabulary linkage by cascade rung", height=520)
fig.show()

occurrences linked across all groups: 59.1%


In [10]:
inst = (
    va.filter(pl.col("status") != "rejected")
    .group_by("data_source", "group")
    .agg(occurrences=pl.len(), resolved=(pl.col("status") == "resolved").mean())
    .collect(engine="streaming")
)
below = inst.filter(pl.col("resolved") < 0.85)
print(f"{len(below)}/{len(inst)} (institution, group) pairs below the 85% occurrence-weighted target")

h = bin_scores(inst["resolved"])
fig = go.Figure()
fig.add_bar(
    x=h["mid"].to_list(), y=h["count"].to_list(), width=0.92 / SCORE_BINS,
    marker=dict(color=CAT[0]), showlegend=False,
    customdata=[[s, s + 1 / SCORE_BINS] for s in h["start"]],
    hovertemplate="%{customdata[0]:.0%}–%{customdata[1]:.0%}<br>%{y} pairs<extra></extra>",
)
fig.add_vline(x=0.85, line=dict(color=MUTED, width=1, dash="dot"), annotation_text="85% target",
              annotation_position="top left", annotation_font=dict(color=MUTED, size=11))
fig.update_xaxes(tickformat=".0%", range=[0, 1], dtick=0.25,
                 title_text="resolved share", title_font=dict(color=MUTED))
fig.update_yaxes(range=[0, h["count"].max() * 1.18],
                 title_text="(institution, field group) pairs", title_font=dict(color=MUTED))
style(fig, "Linkage spread — corpus totals hide the left edge", height=420)
fig.update_layout(bargap=0)
fig.show()

299/465 (institution, group) pairs below the 85% occurrence-weighted target


## 6. Agents (tier 2)

Type routing plus name parsing, occurrence-weighted per field.

In [11]:
pa = pl.scan_parquet(PERSON_ANNOTATIONS)

AGENT_SEGMENTS = ["organisation", "people", "person — inverted", "person — CRF", "deferred"]
AGENT_COLOURS = dict(zip(AGENT_SEGMENTS, CAT + [GRAY_LIGHT]))
agent_segment = (
    pl.when(pl.col("status") == "deferred").then(pl.lit("deferred"))
    .when(pl.col("entity_type") == "organisation").then(pl.lit("organisation"))
    .when(pl.col("entity_type") == "people").then(pl.lit("people"))
    .when(pl.col("sub_component") == "inverted").then(pl.lit("person — inverted"))
    .otherwise(pl.lit("person — CRF"))
)

agent_seg = (
    pa.with_columns(segment=agent_segment)
    .group_by("field_type", "segment").agg(occ=pl.len())
    .with_columns(
        share=pl.col("occ") / pl.col("occ").sum().over("field_type"),
        field_occ=pl.col("occ").sum().over("field_type"),
    )
    .collect(engine="streaming")
)
AGENT_RESOLVED_SHARE = (
    agent_seg.filter(pl.col("segment") != "deferred")["occ"].sum() / agent_seg["occ"].sum()
)
print(f"occurrences routed + parsed: {AGENT_RESOLVED_SHARE:.1%}")

top_fields = (
    agent_seg.unique("field_type").sort("field_occ", descending=True)
    .head(10)["field_type"].to_list()
)
d = agent_seg.filter(pl.col("field_type").is_in(top_fields))
order = (
    d.filter(pl.col("segment") != "deferred")
    .group_by("field_type").agg(pl.col("share").sum()).sort("share")["field_type"].to_list()
)

fig = go.Figure()
for seg in AGENT_SEGMENTS:
    t = d.filter(pl.col("segment") == seg)
    fig.add_bar(y=t["field_type"].to_list(), x=t["share"].to_list(), orientation="h",
                name=seg, marker=dict(color=AGENT_COLOURS[seg]),
                customdata=t["occ"].to_list(),
                hovertemplate="%{y} — " + seg + "<br>%{x:.1%} (%{customdata:,} occ)<extra></extra>")
fig.update_layout(barmode="stack", bargap=0.3)
fig.update_yaxes(categoryorder="array", categoryarray=order)
fig.update_xaxes(tickformat=".0%", range=[0, 1],
                 title_text="occurrence share", title_font=dict(color=MUTED))
style(fig, "Agent routing and parse outcomes — top 10 fields", height=440)
fig.show()

display(agent_seg.pivot(on="segment", index="field_type", values="share")
        .fill_null(0.0).sort("field_type"))

occurrences routed + parsed: 77.8%


field_type,organisation,person — CRF,people,person — inverted,deferred
str,f64,f64,f64,f64,f64
"""spectrum/acquisition_funding_s…",0.560167,0.008022,0.0,0.000349,0.431461
"""spectrum/acquisition_source""",0.133586,0.336396,0.002371,0.247172,0.280475
"""spectrum/associated_event_pers…",0.000432,0.740114,0.0,0.170928,0.088526
"""spectrum/associated_organisati…",0.665421,0.076943,0.004919,0.03552,0.217197
"""spectrum/associated_people""",0.074074,0.358025,0.006173,0.018519,0.54321
…,…,…,…,…,…
"""spectrum/object_production_per…",0.14273,0.232193,0.002021,0.339927,0.283129
"""spectrum/owner""",0.522784,0.071133,0.007501,0.037582,0.360999
"""spectrum/right_holder""",0.493772,0.096581,0.000027,0.009668,0.399952


## 7. Record repair (tiers 1 + 5) — extractable pool and completeness

Refine + novel spans are the extractable pool; completeness is measured on that pool before and after the resolved patch ops.

In [12]:
pc = pl.read_parquet(PROBE_CANDIDATES)

PROBE_SEGMENTS = ["novel", "refine", "additional", "echo", "unparsed_field"]
PROBE_COLOURS = dict(zip(PROBE_SEGMENTS, [CAT[0], CAT[1], CAT[2], GRAY_MID, GRAY_LIGHT]))

probe_seg = (
    pc.group_by("group", "status").agg(occ=pl.len())
    .with_columns(share=pl.col("occ") / pl.col("occ").sum().over("group"))
)
order = (
    probe_seg.filter(pl.col("status").is_in(["novel", "refine"]))
    .group_by("group").agg(pl.col("occ").sum()).sort("occ")["group"].to_list()
)

fig = go.Figure()
for seg in PROBE_SEGMENTS:
    t = probe_seg.filter(pl.col("status") == seg)
    fig.add_bar(y=t["group"].to_list(), x=t["occ"].to_list(), orientation="h",
                name=seg, marker=dict(color=PROBE_COLOURS[seg]),
                customdata=(t["share"] * 100).round(1).to_list(),
                hovertemplate="%{y} — " + seg + "<br>%{x:,} spans (%{customdata}%)<extra></extra>")
fig.update_layout(barmode="stack", bargap=0.3)
fig.update_yaxes(categoryorder="array", categoryarray=order)
fig.update_xaxes(title_text="verified spans", title_font=dict(color=MUTED))
style(fig, "Probe scan — verified spans by comparison class", height=400)
fig.show()

In [13]:
# Destination mapping mirrors 6_record_fixes.
PROD_DATE = "spectrum/object_production_date"
DATE_DEST_BY_SOURCE = {
    "spectrum/acquisition_note": "spectrum/acquisition_date",
    "spectrum/field_collection_note": "spectrum/field_collection_date",
}
PRICE_DEST = "spectrum/object_purchase_price"
DEST = (
    pl.when(pl.col("group") == "__date__")
    .then(pl.col("field_type").replace_strict(DATE_DEST_BY_SOURCE, default=PROD_DATE))
    .when(pl.col("group") == "__price__").then(pl.lit(PRICE_DEST))
    .otherwise(pl.col("group"))
)

pool = (
    pc.filter(pl.col("status").is_in(["refine", "novel"]))
    .with_columns(dest=DEST)
    .group_by("data_source", "record_id", "dest")
    .agg(populated_before=(pl.col("status") == "refine").any())
)
patched = (
    patches.filter((pl.col("task") == "probe_novel") & (pl.col("status") == "resolved"))
    .select("record_id", pl.col("field").alias("dest"))
    .unique()
    .with_columns(patched=pl.lit(True))
)
pool = (
    pool.join(patched, on=["record_id", "dest"], how="left")
    .with_columns(populated_after=pl.col("populated_before") | pl.col("patched").fill_null(False))
)

completeness = (
    pool.group_by("dest")
    .agg(records=pl.len(), before=pl.col("populated_before").mean(),
         after=pl.col("populated_after").mean())
    .sort("records", descending=True)
)
POOL_BEFORE = pool["populated_before"].mean()
POOL_AFTER = pool["populated_after"].mean()
print(f"extractable pool: {len(pool):,} (record, slot) pairs, "
      f"{POOL_BEFORE:.1%} populated before → {POOL_AFTER:.1%} after")

d = completeness.head(10).sort("records")
fig = go.Figure()
fig.add_bar(y=d["dest"].to_list(), x=d["after"].to_list(), orientation="h",
            name="after", marker=dict(color=AFTER),
            hovertemplate="%{y}<br>after %{x:.1%}<extra></extra>")
fig.add_bar(y=d["dest"].to_list(), x=d["before"].to_list(), orientation="h",
            name="before", marker=dict(color=BEFORE),
            hovertemplate="%{y}<br>before %{x:.1%}<extra></extra>")
fig.update_layout(barmode="group")
fig.update_xaxes(tickformat=".0%", range=[0, 1],
                 title_text="share of pool records with the slot populated",
                 title_font=dict(color=MUTED))
style(fig, "Completeness on the extractable pool — before/after patches", height=420)
fig.update_layout(legend_traceorder="reversed")
fig.show()

extractable pool: 379,305 (record, slot) pairs, 44.2% populated before → 93.6% after


In [14]:
# Tier-5 LLM extraction ran on the admission-queue test subset only.
llm_patches = patches.filter(pl.col("sub_component") == "llm")
accepted = llm_patches.filter(pl.col("status") == "resolved")
total_tokens = llm_raw["prompt_tokens"].sum() + llm_raw["completion_tokens"].sum()
print(f"admitted records: {len(llm_raw):,}; accepted ops: {len(accepted):,} across "
      f"{accepted['record_id'].n_unique():,} records")
print(f"tokens/record: {total_tokens / len(llm_raw):,.0f}   "
      f"tokens/accepted op: {total_tokens / max(len(accepted), 1):,.0f}")
display(llm_patches.group_by("task", "status").len().sort("task", "len", descending=[False, True]))

admitted records: 5,996; accepted ops: 705,664 across 388,955 records
tokens/record: 1,053   tokens/accepted op: 9


task,status,len
str,str,u32
null,"""rejected""",73627
null,"""deferred""",210
"""extract_date""","""resolved""",63492
"""extract_dimension""","""resolved""",71523
"""extract_material""","""resolved""",550228
"""relocate""","""resolved""",20421


## 8. Marginal quality per unit cost

Occurrences improved per Wh per stage. Energy is all logged runs, so rates are lower bounds; the LLM stages ran on admission subsets.

In [15]:
# Resolved and flagged cost the same energy, so count both
va_decided = (
    va.filter(pl.col("status").is_in(["resolved", "flagged"]))
    .group_by("tier", "status", rerank=pl.col("sub_component") == "rerank")
    .agg(occ=pl.len())
    .collect(engine="streaming")
)
vt = {
    (t, s, r): o for t, s, r, o in va_decided.iter_rows()
}
resolved = lambda tier, rr=False: vt.get((tier, "resolved", rr), 0)  # noqa: E731
flagged = lambda tier, rr=False: vt.get((tier, "flagged", rr), 0)  # noqa: E731
person_resolved = (
    pa.filter(pl.col("status") == "resolved").select(pl.len())
    .collect(engine="streaming").item()
)
pool_spans = pc.filter(pl.col("status").is_in(["refine", "novel"])).height
mech_resolved = patches.filter(
    (pl.col("sub_component") == "mechanical") & (pl.col("status") == "resolved")).height
mech_flagged = patches.filter(
    (pl.col("sub_component") == "mechanical") & (pl.col("status") == "flagged")).height

marginal = pl.DataFrame({
    "stage": ["probe scan + verify", "vocab — exact + fuzzy", "vocab — semantic",
              "vocab — rerank", "vocab — LLM atomiser", "agents — route + parse",
              "record fixes — mechanical", "record fixes — LLM"],
    "tier": [1, 2, 3, 4, 4, 2, 1, 5],
    # The probe stage only surfaces candidates, none applied here
    "resolved_occ": [0, resolved(2), resolved(3), 0, resolved(4),
                     person_resolved, mech_resolved, len(accepted)],
    "flagged_occ": [pool_spans, flagged(2), flagged(3), flagged(4, rr=True),
                    flagged(4), 0, mech_flagged, 0],
    "energy_wh": [probe_wh,
                  wh["vocab — authority indexes"] + wh["vocab — exact + fuzzy"],
                  wh["vocab — semantic"], wh["vocab — rerank"],
                  wh["vocab — LLM atomiser"],
                  wh["agents — route + parse"], wh["record fixes — mechanical ops"],
                  wh["record fixes — LLM extraction"]],
}).with_columns(
    decided_occ=pl.col("resolved_occ") + pl.col("flagged_occ"),
).with_columns(
    resolved_per_wh=(pl.col("resolved_occ") / pl.col("energy_wh")).round(1),
    decided_per_wh=(pl.col("decided_occ") / pl.col("energy_wh")).round(1),
)
display(marginal.sort("decided_per_wh", descending=True))

d = marginal.filter(pl.col("decided_per_wh") > 0).sort("decided_per_wh")
fig = go.Figure()
# Hollow marker shows decisions made, the gap is review queue
fig.add_scatter(
    y=d["stage"].to_list(), x=d["decided_per_wh"].to_list(), mode="markers",
    name="decided (applied + flagged)", marker=dict(color=INK, size=11),
    customdata=d["decided_occ"].to_list(),
    hovertemplate="%{y}<br>%{x:,.1f} occ/Wh decided (%{customdata:,})<extra></extra>",
)
applied = d.filter(pl.col("resolved_per_wh") > 0)
fig.add_scatter(
    y=applied["stage"].to_list(), x=applied["resolved_per_wh"].to_list(), mode="markers",
    name="applied only", marker=dict(color=SURFACE, size=11,
                                     line=dict(color=INK, width=1.5)),
    customdata=applied["resolved_occ"].to_list(),
    hovertemplate="%{y}<br>%{x:,.1f} occ/Wh applied (%{customdata:,})<extra></extra>",
)
fig.update_yaxes(categoryorder="array", categoryarray=d["stage"].to_list())
fig.update_xaxes(type="log", title_text="occurrences decided per Wh (log)",
                 title_font=dict(color=MUTED))
style(fig, "Marginal quality per unit energy — applied against all decisions", height=400)
fig.show()

stage,tier,resolved_occ,flagged_occ,energy_wh,decided_occ,resolved_per_wh,decided_per_wh
str,i64,i64,i64,f64,i64,f64,f64
"""vocab — exact + fuzzy""",2,21375755,6717876,1.307,28093631,1.6354824e7,2.1495e7
"""record fixes — mechanical""",1,533914,157376,0.089,691290,5999033.7,7767303.4
"""agents — route + parse""",2,4242876,0,2.967,4242876,1430022.2,1430022.2
"""vocab — LLM atomiser""",4,331001,492596,108.951,823597,3038.1,7559.3
"""probe scan + verify""",1,0,577612,76.468,577612,0.0,7553.6
"""vocab — rerank""",4,0,881143,960.859,881143,0.0,917.0
"""record fixes — LLM""",5,705664,0,7040.791,705664,100.2,100.2
"""vocab — semantic""",3,0,0,24.008,0,0.0,0.0


## 9. Method experiments (X4)

E1–E3 re-ran the atomiser, extraction prompt and edit representation choices on frozen gold (`experiments/README.md`). Results read from `analysis_output/experiments/`, numbers as of 2026-07-15. E1: the deterministic atomiser beats gpt-oss-20b on occurrence-weighted exact match (0.613 vs 0.461) at ~1/125th the energy. E2: task-contract v2 lifts strict P/R to 0.62/0.48 composed and 0.60/0.53 single-task; Qwen3-1.7B collapses. E3: JSON-patch has the best recall (0.39); `diff`'s precision is a low-yield artefact.

In [16]:

atomiser = pl.read_parquet(EXPERIMENTS_PATH / "atomiser_variants" / "results.parquet")
display(atomiser.select("variant", "decision_acc", "atom_precision", "atom_recall", "atom_f1",
                  "ht_exact_match", "wh_per_1k_values", "projected_queue_wh")
        .sort("ht_exact_match", descending=True))

extraction = (
    pl.read_parquet(EXPERIMENTS_PATH / "extraction_results.parquet")
    .with_columns(
        f1=(2 * pl.col("precision_strict") * pl.col("recall_strict")
            / (pl.col("precision_strict") + pl.col("recall_strict"))).round(4),
    )
)
best = extraction.sort("f1", descending=True).row(0, named=True)
print(f"scaling configuration: {best['variant']} — "
      f"P {best['precision_strict']:.2f} / R {best['recall_strict']:.2f}, "
      f"{best['tokens_per_accepted_op']:,.0f} tokens/accepted op, "
      f"~{best['projected_queue_wh'] / 1e3:.1f} kWh projected over the full queue")
display(extraction.select("exp", "variant", "precision_strict", "recall_strict", "f1",
                          "placement_error", "tokens_per_accepted_op", "wh_per_record",
                          "projected_queue_wh").sort("f1", descending=True))

fig = go.Figure()
fig.add_scatter(
    x=atomiser["wh_per_1k_values"].to_list(), y=atomiser["ht_exact_match"].to_list(),
    mode="markers+text", text=atomiser["variant"].to_list(), textposition="top center",
    textfont=dict(size=11, color=INK2), marker=dict(color=CAT[0], size=11),
    showlegend=False,
    hovertemplate="%{text}<br>%{x:.3f} Wh/1k values, HT exact match %{y:.3f}<extra></extra>",
)
fig.update_xaxes(type="log", title_text="Wh per 1k values (log)", title_font=dict(color=MUTED))
fig.update_yaxes(title_text="occurrence-weighted exact match", title_font=dict(color=MUTED),
                 rangemode="tozero")
style(fig, "atomiser accuracy vs energy", height=400)
fig.show()

EXP_NAMES = {"extraction_variants": "model × task contract", "representation_variants": "representation"}
fig = go.Figure()
for i, (exp, label) in enumerate(EXP_NAMES.items()):
    t = extraction.filter(pl.col("exp") == exp)
    fig.add_scatter(
        x=t["wh_per_record"].to_list(), y=t["f1"].to_list(),
        mode="markers+text", name=label, text=t["variant"].to_list(),
        textposition="top center", textfont=dict(size=11, color=INK2),
        marker=dict(color=CAT[i], size=11),
        hovertemplate="%{text}<br>%{x:.4f} Wh/record, F1 %{y:.3f}<extra></extra>",
    )
fig.update_xaxes(type="log", title_text="Wh per record (log)", title_font=dict(color=MUTED))
fig.update_yaxes(title_text="strict F1 (pooled gold)", title_font=dict(color=MUTED),
                 rangemode="tozero")
style(fig, "extraction quality vs energy", height=440)
fig.show()

variant,decision_acc,atom_precision,atom_recall,atom_f1,ht_exact_match,wh_per_1k_values,projected_queue_wh
str,f64,f64,f64,f64,f64,f64,f64
"""deterministic""",0.71855,0.519663,0.512465,0.516039,0.61277,0.040226,8.789698
"""gpt-oss-20b:everywhere""",0.707889,0.608939,0.805171,0.693439,0.518505,4.15398,1181.458234
"""lfm2.5-350m""",0.69936,0.439942,0.561404,0.493306,0.504138,0.491651,107.428762
"""gpt-oss-20b""",0.703625,0.497789,0.727608,0.591148,0.453151,5.234744,1143.822967
"""lfm2.5-8b-a1b""",0.652452,0.377805,0.575254,0.456076,0.335074,15.503652,3387.640997
"""qwen3-1.7b""",0.639659,0.384572,0.630656,0.477789,0.324621,243.220607,53145.161895


scaling configuration: gpt-oss-20b:single_task_v2 — P 0.64 / R 0.56, 1,928 tokens/accepted op, ~10.3 kWh projected over the full queue


exp,variant,precision_strict,recall_strict,f1,placement_error,tokens_per_accepted_op,wh_per_record,projected_queue_wh
str,str,f64,f64,f64,f64,f64,f64,f64
"""extraction_variants""","""gpt-oss-20b:single_task_v2""",0.643357,0.564417,0.6013,0.0,1928.257732,0.010033,10250.689394
"""extraction_variants""","""gpt-oss-20b:composed_v2""",0.612,0.469325,0.5312,0.0,1435.657371,0.007124,7279.330543
"""extraction_variants""","""gpt-oss-20b:single_task""",0.493289,0.45092,0.4712,0.0,1576.154362,0.009782,9995.139153
"""extraction_variants""","""gpt-oss-20b:composed""",0.483108,0.43865,0.4598,0.0,991.986532,0.00862,8807.167697
"""representation_variants""","""rep:json_patch""",0.465574,0.435583,0.4501,0.0,952.974026,0.007143,7298.587547
"""representation_variants""","""rep:fields_only""",0.446154,0.355828,0.3959,0.0,1035.762963,0.005992,6122.608123
"""representation_variants""","""rep:diff""",0.553191,0.239264,0.334,0.0,1959.0,0.006141,6274.156874
"""extraction_variants""","""qwen3-1.7b:composed""",0.26087,0.092025,0.1361,0.032258,2940.418803,0.090142,92101.53078
"""extraction_variants""","""qwen3-1.7b:single_task""",0.161972,0.070552,0.0983,0.115385,3428.269737,0.089787,91738.989767


## 10. Spread across institutions

Each axis recomputed per institution and pooled into a distribution. No institution is named or ranked; institutions under `MIN_OCC` occurrences on an axis are dropped.

In [17]:
MIN_OCC = 100

date_verdicts = pl.DataFrame({"value": list(parses), "parses": list(parses.values())})
date_inst = (
    fs.filter(pl.col("field_type").is_in(list(date_fields())))
    .group_by("data_source", "value").agg(count=pl.len())
    .collect(engine="streaming")
    .join(date_verdicts, on="value", how="left")
    .group_by("data_source")
    .agg(occurrences=pl.col("count").sum(),
         score=(pl.col("parses") * pl.col("count")).sum() / pl.col("count").sum())
)
vocab_inst = (
    va.filter(pl.col("status") != "rejected")
    .group_by("data_source").agg(occurrences=pl.len(), score=(pl.col("status") == "resolved").mean())
    .collect(engine="streaming")
)
agent_inst = (
    pa.group_by("data_source").agg(occurrences=pl.len(), score=(pl.col("status") != "deferred").mean())
    .collect(engine="streaming")
)
pool_inst = pool.group_by("data_source").agg(occurrences=pl.len(), score=pl.col("populated_after").mean())

CORPUS = {
    "date occurrences in EDTF form": DATE_AFTER,
    "vocabulary occurrences linked": VOCAB_RESOLVED_SHARE,
    "agent occurrences typed + parsed": AGENT_RESOLVED_SHARE,
    "extractable-pool slots populated": POOL_AFTER,
}
spread = pl.concat(
    [f.with_columns(metric=pl.lit(m)).select("metric", "occurrences", "score")
     for m, f in zip(CORPUS, [date_inst, vocab_inst, agent_inst, pool_inst])]
).filter(pl.col("occurrences") >= MIN_OCC)
binned = {m: bin_scores(spread.filter(pl.col("metric") == m)["score"]) for m in CORPUS}

fig = make_subplots(rows=2, cols=2, subplot_titles=list(CORPUS),
                    vertical_spacing=0.16, horizontal_spacing=0.09)
for i, (metric, corpus) in enumerate(CORPUS.items()):
    row, col = divmod(i, 2)
    h = binned[metric]
    top = h["count"].max() * 1.28
    fig.add_bar(
        x=h["mid"].to_list(), y=h["count"].to_list(), width=0.92 / SCORE_BINS,
        marker=dict(color=AFTER), showlegend=False,
        customdata=[[s, s + 1 / SCORE_BINS] for s in h["start"]],
        hovertemplate="%{customdata[0]:.0%}–%{customdata[1]:.0%}<br>%{y} institutions<extra></extra>",
        row=row + 1, col=col + 1,
    )
    fig.add_vline(x=corpus, line=dict(color=MUTED, width=1, dash="dot"), row=row + 1, col=col + 1)
    fig.add_annotation(x=corpus, y=top, text=f"corpus {corpus:.0%}", showarrow=False,
                       xanchor="right" if corpus > 0.5 else "left", xshift=-5 if corpus > 0.5 else 5,
                       yanchor="top", font=dict(color=MUTED, size=11), row=row + 1, col=col + 1)
    fig.update_yaxes(range=[0, top], row=row + 1, col=col + 1)

fig.update_xaxes(tickformat=".0%", range=[0, 1], dtick=0.25)
fig.update_yaxes(title_text="institutions", title_font=dict(color=MUTED), col=1)
for a in fig.layout.annotations[:len(CORPUS)]:
    a.font = dict(size=12, color=INK2)
style(fig, "Score spread across institutions", height=560)
fig.update_layout(bargap=0)
fig.show()

spread_summary = (
    spread.join(pl.DataFrame({"metric": list(CORPUS), "corpus": list(CORPUS.values())}), on="metric")
    .group_by("metric")
    .agg(institutions=pl.len(), p10=pl.col("score").quantile(0.1), median=pl.col("score").median(),
         p90=pl.col("score").quantile(0.9), corpus=pl.col("corpus").first(),
         below_corpus=(pl.col("score") < pl.col("corpus")).mean())
    .sort("median")
)
institution_spread = pl.concat(
    [h.with_columns(metric=pl.lit(m)) for m, h in binned.items()]
).select("metric", pl.col("start").alias("bin_start"), pl.col("count").alias("institutions"))
institution_spread.write_parquet(OUT_DIR / "institution_spread.parquet")
spread_summary.write_parquet(OUT_DIR / "institution_spread_summary.parquet")
spread_summary

metric,institutions,p10,median,p90,corpus,below_corpus
str,u32,f64,f64,f64,f64,f64
"""vocabulary occurrences linked""",101,0.222014,0.529103,0.757885,0.591271,0.613861
"""agent occurrences typed + pars…",93,0.409615,0.8771,0.978604,0.778268,0.354839
"""date occurrences in EDTF form""",83,0.802198,0.973913,0.999828,0.910806,0.277108
"""extractable-pool slots populat…",78,0.893617,0.988065,1.0,0.93578,0.141026


## 11. Before/after summary

One occurrence-weighted number per quality axis.

In [18]:
summary = pl.DataFrame({
    "metric": [
        "text free of mojibake",
        "date occurrences in EDTF form",
        "date head-20 pattern share (consistency)",
        "vocabulary occurrences linked",
        "agent occurrences typed + parsed",
        "extractable-pool slots populated",
    ],
    "before": [1 - MOJIBAKE_SHARE, DATE_BEFORE, CONSISTENCY_BEFORE, 0.0, 0.0, POOL_BEFORE],
    "after": [1.0, DATE_AFTER, CONSISTENCY_AFTER, VOCAB_RESOLVED_SHARE,
              AGENT_RESOLVED_SHARE, POOL_AFTER],
}).with_columns(delta=(pl.col("after") - pl.col("before")).round(4))

fig = go.Figure()
for i, (metric, before, after, _) in enumerate(summary.iter_rows()):
    fig.add_scatter(x=[before, after], y=[metric, metric], mode="lines",
                    line=dict(color=GRID, width=2), showlegend=False, hoverinfo="skip")
    fig.add_scatter(x=[before], y=[metric], mode="markers", name="before",
                    marker=dict(color=BEFORE, size=11), showlegend=(i == 0),
                    hovertemplate="before %{x:.1%}<extra></extra>")
    fig.add_scatter(x=[after], y=[metric], mode="markers", name="after",
                    marker=dict(color=AFTER, size=11), showlegend=(i == 0),
                    hovertemplate="after %{x:.1%}<extra></extra>")
fig.update_xaxes(tickformat=".0%", range=[-0.02, 1.05],
                 title_text="occurrence-weighted share", title_font=dict(color=MUTED))
fig.update_yaxes(autorange="reversed")
style(fig, "Quality before → after, per axis", height=380)
fig.show()

stage_costs.write_parquet(OUT_DIR / "stage_costs.parquet")
workload.write_parquet(OUT_DIR / "workload.parquet")
marginal.write_parquet(OUT_DIR / "marginal_gain.parquet")
summary.write_parquet(OUT_DIR / "quality_summary.parquet")
summary

metric,before,after,delta
str,f64,f64,f64
"""text free of mojibake""",0.999867,1.0,0.0001
"""date occurrences in EDTF form""",0.300919,0.910806,0.6099
"""date head-20 pattern share (co…",0.783142,0.979941,0.1968
"""vocabulary occurrences linked""",0.0,0.591271,0.5913
"""agent occurrences typed + pars…",0.0,0.778268,0.7783
"""extractable-pool slots populat…",0.441655,0.93578,0.4941


Artefacts land in `analysis_output/evaluation/`. Limitations: the emissions log keeps development re-runs, the LLM stages were measured on subsets (`evaluation/llm_queue_projections.json`), and financial cost is electricity only.